In [ ]:
# Unsloth install
!pip install -q unsloth

# Restart runtime immediately after install
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from unsloth import FastLanguageModel

print("UNSLOTH WORKING")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
UNSLOTH WORKING


In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
Tesla T4


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = 512,
    load_in_4bit = True,
)

print("MODEL LOADED")

==((====))==  Unsloth 2026.6.1: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


MODEL LOADED


In [ ]:
import json
import pandas as pd
from datasets import load_dataset

mentalchat = load_dataset("ShenLab/MentalChat16K")
esconv = load_dataset("thu-coai/esconv")

records = []

# --------------------
# MentalChat16K
# --------------------

for row in mentalchat["train"]:

    records.append({
        "instruction": row["instruction"],
        "input": row["input"],
        "output": row["output"]
    })

# --------------------
# ESConv
# --------------------

strategy_map = {

    "Reflection of feelings":
    "<STRATEGY_REFLECTION>",

    "Affirmation and Reassurance":
    "<STRATEGY_AFFIRMATION>",

    "Question":
    "<STRATEGY_QUESTION>",

    "Providing Suggestions":
    "<STRATEGY_SUGGESTION>",

    "Restatement or Paraphrasing":
    "<STRATEGY_PARAPHRASE>",

    "Information":
    "<STRATEGY_INFORMATION>"
}

for sample in esconv["train"]:

    try:

        conversation = json.loads(
            sample["text"]
        )

        dialog = conversation["dialog"]

        user_msgs = []
        final_response = None
        strategy_token = "<STRATEGY_GENERAL>"

        for turn in dialog:

            if turn["speaker"] == "usr":

                user_msgs.append(
                    turn["text"]
                )

            elif turn["speaker"] == "sys":

                final_response = turn["text"]

                if "strategy" in turn:

                    strategy_token = strategy_map.get(
                        turn["strategy"],
                        "<STRATEGY_GENERAL>"
                    )

        if (
            len(user_msgs) > 0
            and final_response is not None
        ):

            records.append({

                "instruction":
                strategy_token,

                "input":
                "\n".join(user_msgs),

                "output":
                final_response
            })

    except:
        pass

df = pd.DataFrame(records)

print("Total Samples:", len(df))

print(df.tail())

df.to_json(
    "ember_train_final.json",
    orient="records",
    indent=2
)

print("Saved!")

README.md:   0%|          | 0.00/3.95k [00:00<?, ?B/s]

Interview_Data_6K.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Synthetic_Data_10K.csv:   0%|          | 0.00/32.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/510 [00:00<?, ?B/s]

...

test.txt:   0%|          | 0.00/881k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/910 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/195 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/195 [00:00<?, ? examples/s]

Total Samples: 16994
                  instruction  \
16989      <STRATEGY_GENERAL>   
16990   <STRATEGY_SUGGESTION>   
16991      <STRATEGY_GENERAL>   
16992  <STRATEGY_AFFIRMATION>   
16993      <STRATEGY_GENERAL>   

                                                   input  \
16989  hi\nI'm in a big mess\nI am not fine , I tried...   
16990  Hi, how are you doing today?\nYes I am. How ar...   
16991  hello\nso am having a hard time with my friend...   
16992  hello\nstressed out, anxious , this covid life...   
16993  I'm so sad\nMy partner left me for another wom...   

                                                  output  
16989                                          good luck  
16990  The study suggests that the story be upbeat an...  
16991                                       Same to you!  
16992  I understand that feeling! well hopefully we g...  
16993                   I'm so happy I was able to help.  
Saved!


In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import pandas as pd

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_json("ember_train_final.json")

print("Total Samples:", len(df))

dataset = Dataset.from_pandas(df)

# =====================================================
# FORMAT DATASET
# =====================================================

def formatting_prompts_func(examples):

    texts = []

    for instruction, user_input, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):

        text = f"""### Instruction:
{instruction}

### User:
{user_input}

### Therapist:
{output}"""

        texts.append(text)

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True
)

# =====================================================
# LOAD MODEL
# =====================================================

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# =====================================================
# LORA CONFIG
# =====================================================

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,

    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha = 16,
    lora_dropout = 0,

    bias = "none",

    use_gradient_checkpointing = "unsloth",
)

# =====================================================
# TRAIN / VALIDATION SPLIT
# =====================================================

dataset = dataset.train_test_split(
    test_size = 0.05,
    seed = 42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Train Samples:", len(train_dataset))
print("Eval Samples :", len(eval_dataset))

# =====================================================
# TRAINER
# =====================================================

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,

    dataset_text_field = "text",

    args = TrainingArguments(

        output_dir = "ember_x",

        num_train_epochs = 1,

        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        learning_rate = 2e-4,

        warmup_steps = 50,

        logging_steps = 25,

        eval_strategy = "steps",
        eval_steps = 500,

        save_strategy = "steps",
        save_steps = 500,

        save_total_limit = 2,

        load_best_model_at_end = True,

        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        optim = "adamw_8bit",

        weight_decay = 0.01,

        fp16 = True,

        report_to = "none",
    ),
)

# =====================================================
# TRAIN
# =====================================================

print("\nStarting Training...\n")

trainer_stats = trainer.train()

# =====================================================
# SAVE FINAL MODEL
# =====================================================

model.save_pretrained(
    "ember_x_gemma_lora"
)

tokenizer.save_pretrained(
    "ember_x_gemma_lora"
)

print("\nModel Saved!")

# =====================================================
# ZIP MODEL
# =====================================================

!zip -r ember_x_gemma_lora.zip ember_x_gemma_lora

print("\nZIP FILE CREATED!")

# =====================================================
# TRAINING SUMMARY
# =====================================================

print("\nTraining Stats:")
print(trainer_stats)

Total Samples: 16994


Map:   0%|          | 0/16994 [00:00<?, ? examples/s]

==((====))==  Unsloth 2026.6.1: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


Train: 16144
Eval : 850


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/16144 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/850 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,144 | Num Epochs = 2 | Total steps = 2,018
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.

Step,Training Loss
25,26.012139
50,21.030696
75,10.031752
100,7.034232
125,6.425418
150,6.124703
175,5.892570
200,5.739686
225,5.592742
250,5.483074
